# A bare-corpus BERTopic *stack* — and what FoodOn actually buys

**Question.** If you had only the chunk texts (no ontology), could BERTopic alone produce
something shaped like FoodScholar's **Layer A backbone + Layer B themes** — and how much of the
*FoodOn structure* does text-alone recover?

**What we build.** A two-tier BERTopic "stack" from the bare corpus:
- **Themes** = BERTopic's leaf topics (the Layer B analog).
- **Backbone** = a coarse agglomerative grouping of those topics into macro-shelves, plus
  BERTopic's own `hierarchical_topics` tree (the Layer A analog).

**The ablation.** We then score that stack against the *FoodOn grounding* that lives in the bare
corpus (`foodon_ids` per chunk): at the **leaf** level (dominant FoodOn id) and at a **coarse**
level (the FoodOn *ancestor* of that id — a Layer-A-shaped bucket derived straight from the
ontology's is-a edges). High recovery ⇒ the ontology is mostly re-derivable from text; low
recovery ⇒ FoodOn contributes structure text can't find. **No library pipeline, no LLM** — reuses
the cached BGE-base vectors from `bertopic_explore.ipynb`.

> This is the companion to `bertopic_layerb_headtohead.ipynb`, which runs the *real* Layer B and
> compares per shelf. Here we stay bare-corpus and ask the ontology-value question.

## 1 · Setup & aligned data

In [ ]:
try:
    import bertopic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "bertopic", "umap-learn", "plotly"], check=True)

import json, math
from collections import Counter
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE = ROOT / "data" / "cache"
OUT = ROOT / "notebooks" / "out"; OUT.mkdir(parents=True, exist_ok=True)
SNAPSHOT = ROOT / "data" / "annotated.parquet"
FOODON_CACHE = ROOT / "data" / "foodon_cache.parquet"
RANDOM_STATE = 42
K_MACRO = 12      # coarse backbone tier: number of macro-shelves
ANCESTOR_DEPTH = 3  # FoodOn depth used for the coarse ontology reference

# Load the cached embeddings + the exact chunk-id order they were built with.
emb_files = sorted(CACHE.glob("bge_base_emb_*.npy"))
assert emb_files, "Run bertopic_explore.ipynb first to build data/cache/bge_base_emb_*.npy"
EMB_PATH = emb_files[-1]
IDS_PATH = CACHE / EMB_PATH.name.replace("emb", "ids").replace(".npy", ".json")
embeddings = np.load(EMB_PATH)
chunk_ids = json.loads(IDS_PATH.read_text())

# Align the corpus dataframe to the cached id order (guarantees row↔vector match).
df = pd.read_parquet(SNAPSHOT).set_index("chunk_id").loc[chunk_ids].reset_index()
docs = df["text"].tolist()
assert len(docs) == embeddings.shape[0] == len(chunk_ids)
print(f"{len(docs):,} chunks aligned to {embeddings.shape} embeddings  (cache: {EMB_PATH.name})")

## 2 · Build the stack — themes (leaf) + backbone (coarse)

In [ ]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer

topic_model = BERTopic(
    umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                    metric="cosine", random_state=RANDOM_STATE),
    hdbscan_model=HDBSCAN(min_cluster_size=30, metric="euclidean",
                          cluster_selection_method="eom", prediction_data=True),
    vectorizer_model=CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=5),
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    calculate_probabilities=False, verbose=True,
)
topics, _ = topic_model.fit_transform(docs, embeddings=embeddings)
topics = np.asarray(topics)
info = topic_model.get_topic_info()
topic_ids = [t for t in info.Topic if t != -1]
print(f"themes (leaf topics): {len(topic_ids)} | outliers: {np.mean(topics == -1):.1%}")

In [ ]:
# Coarse backbone: agglomerate the leaf topics' c-TF-IDF profiles into K_MACRO macro-shelves.
from sklearn.cluster import AgglomerativeClustering

ctfidf = topic_model.c_tf_idf_                      # (n_topics_incl_outlier) x vocab, sparse
row_of = {t: i for i, t in enumerate(topic_model.topic_labels_.keys())}  # topic -> ctfidf row
X_topics = np.asarray([ctfidf[row_of[t]].toarray()[0] for t in topic_ids])
macro_of_topic = dict(zip(topic_ids, AgglomerativeClustering(
    n_clusters=min(K_MACRO, len(topic_ids)), metric="cosine", linkage="average"
).fit_predict(X_topics)))

# per-chunk labels for the two tiers (outliers carry -1 at both)
df["topic"] = topics
df["macro"] = df["topic"].map(lambda t: macro_of_topic.get(t, -1))
print(f"backbone macro-shelves: {len(set(macro_of_topic.values()))}")

# BERTopic's own inferred backbone (qualitative Layer-A analog)
hier = topic_model.hierarchical_topics(docs)
print(topic_model.get_topic_tree(hier))

## 3 · The FoodOn reference (derived from the bare corpus)

For each chunk we take its **dominant FoodOn id** (leaf reference) and that id's **ancestor at
`ANCESTOR_DEPTH`** in FoodOn's is-a tree (coarse reference, depth 3 by default). Depth and
ancestors come straight from `foodon_cache.parquet`'s `parent_ids` — no pipeline needed.

In [ ]:
fo = pd.read_parquet(FOODON_CACHE)
parent = {i: (p if p is not None else []) for i, p in zip(fo["id"], fo["parent_ids"])}
fo_label = dict(zip(fo["id"], fo["label"]))
is_foodon = lambda i: isinstance(i, str) and i.startswith("FOODON:")
lbl = lambda i: fo_label.get(i, i)

_depth = {}
def depth(i, seen=frozenset()):
    if i in _depth: return _depth[i]
    if i in seen: return 0
    ps = [p for p in parent.get(i, []) if is_foodon(p)]
    d = 0 if not ps else 1 + min(depth(p, seen | {i}) for p in ps)
    _depth[i] = d; return d

def ancestor_at(i, k):
    cur, guard = i, 0
    while depth(cur) > k and guard < 50:
        ps = [p for p in parent.get(cur, []) if is_foodon(p)]
        if not ps: break
        cur = min(ps, key=depth); guard += 1
    return cur

df["fo_dom"] = df["foodon_ids"].apply(lambda x: x[0] if (x is not None and len(x) > 0) else None)
df["fo_anc"] = df["fo_dom"].apply(lambda i: ancestor_at(i, ANCESTOR_DEPTH) if is_foodon(i) else None)
print("example dominant-id → depth-%d ancestor:" % ANCESTOR_DEPTH)
for i in df["fo_dom"].dropna().unique()[:6]:
    a = ancestor_at(i, ANCESTOR_DEPTH)
    print(f"  {lbl(i)} (d{depth(i)}) → {lbl(a)}")

## 4 · Ablation metrics — how much does the stack recover?

In [ ]:
from sklearn.metrics import (normalized_mutual_info_score as NMI,
                             adjusted_rand_score as ARI, v_measure_score)

def npmi_coherence(model, ids, topn=10):
    tw = {t: [w for w, _ in model.get_topic(t)[:topn]] for t in ids}
    vocab = sorted({w for ws in tw.values() for w in ws})
    Xb = CountVectorizer(vocabulary=vocab, binary=True, ngram_range=(1, 2)).fit_transform(docs).tocsc()
    D = Xb.shape[0]; col = {w: j for j, w in enumerate(vocab)}
    dfc = np.asarray(Xb.sum(axis=0)).ravel()
    def _n(a, b):
        da, db = dfc[col[a]], dfc[col[b]]
        if da == 0 or db == 0: return 0.0
        co = Xb[:, col[a]].multiply(Xb[:, col[b]]).nnz
        if co == 0: return -1.0
        pa, pb, pab = da/D, db/D, co/D
        return math.log(pab/(pa*pb)) / (-math.log(pab))
    vals = [np.mean([_n(a, b) for a, b in combinations(tw[t], 2)]) for t in ids if len(tw[t]) > 1]
    return float(np.nanmean(vals))

leaf = df[df["fo_dom"].notna() & (df["topic"] != -1)]
coarse = df[df["fo_anc"].notna() & (df["macro"] != -1)]

rows = [
    ("themes (leaf topics)", len(topic_ids)),
    ("macro-shelves (backbone)", len(set(macro_of_topic.values()))),
    ("outlier share", f"{np.mean(df['topic'] == -1):.1%}"),
    ("mean NPMI coherence", f"{npmi_coherence(topic_model, topic_ids):.3f}"),
    ("LEAF recovery  NMI(topic ; FoodOn id)", f"{NMI(leaf['fo_dom'], leaf['topic']):.3f}"),
    ("COARSE recovery NMI(macro ; FoodOn ancestor)", f"{NMI(coarse['fo_anc'], coarse['macro']):.3f}"),
    ("COARSE recovery ARI(macro ; FoodOn ancestor)", f"{ARI(coarse['fo_anc'], coarse['macro']):.3f}"),
    ("COARSE recovery V-measure", f"{v_measure_score(coarse['fo_anc'], coarse['macro']):.3f}"),
]
summary = pd.DataFrame(rows, columns=["metric", "value"])
summary

In [ ]:
# Coverage: BERTopic drops outliers; show coverage-by-construction is recoverable via reduce_outliers.
reduced = np.asarray(topic_model.reduce_outliers(docs, list(topics)))
print(f"coverage before: {np.mean(topics != -1):.1%}   after reduce_outliers: {np.mean(reduced != -1):.1%}")

## 5 · Qualitative — the inferred backbone next to FoodOn

In [ ]:
# Each macro-shelf: its member topics' words and the FoodOn entities (+ ancestor labels) underneath.
rows = []
for m in sorted(set(macro_of_topic.values())):
    mt = [t for t in topic_ids if macro_of_topic[t] == m]
    sub = df[df["macro"] == m]
    words = Counter()
    for t in mt:
        words.update(dict(topic_model.get_topic(t)[:5]))
    fo_anc = Counter(a for a in sub["fo_anc"] if a is not None)
    rows.append({
        "macro": m, "n_topics": len(mt), "chunks": len(sub),
        "top_words": ", ".join(w for w, _ in words.most_common(6)),
        "top_foodon_ancestors": "; ".join(f"{lbl(a)} ({n})" for a, n in fo_anc.most_common(4)),
    })
pd.DataFrame(rows)

## 6 · Reading the ablation

- **LEAF recovery** (`NMI(topic ; FoodOn id)`): how much BERTopic's themes line up with *which food*
  a chunk is about. From `bertopic_explore.ipynb` this was ~0.36 — moderate: text groups by concept,
  not food, so it only partly recovers food identity.
- **COARSE recovery** (`macro ; FoodOn ancestor`): the headline ablation number. If the data-driven
  backbone matches FoodOn's is-a tiers, the ontology adds little *structure*; if it's low, FoodOn's
  backbone is genuinely non-derivable from text and earns its place.
- **Coverage**: BERTopic drops a chunk-share to outliers, but `reduce_outliers` recovers near-full
  coverage — so coverage-by-construction is *achievable* in a BERTopic stack, it's just not the default.
- **What text can't give you regardless of the numbers:** stable FoodOn ids, is-a semantics, and
  provenance. Even at high recovery, the BERTopic backbone is keyword-labeled and unanchored — the
  ontology's contribution is *traceability*, not just *grouping*.

Compare these recovery numbers against the per-shelf head-to-head in `bertopic_layerb_headtohead.ipynb`,
where BERTopic is matched against the *actual* Layer B themes (which also use the entity signal this
notebook's BERTopic ignores).